# Sturm-Liouville Theory
## The framework behind separation of variables

When you separate variables in any **linear PDE** on a rectangular domain, each
variable produces an ODE boundary value problem of the form:

$$\boxed{-\frac{d}{dx}\!\left[p(x)\,y'\right] + q(x)\,y = \lambda\,w(x)\,y, \quad x \in [a,b]}$$

with boundary conditions at $a$ and $b$. This is the **Sturm-Liouville (S-L) problem**.

| Parameter | Role |
|-----------|------|
| $p(x) > 0$ | "stiffness" — weights the derivative term |
| $q(x)$ | potential / restoring force |
| $w(x) > 0$ | **weight function** (also called density) |
| $\lambda$ | eigenvalue — the separation constant |

The solution $y(x)$ is the **eigenfunction** associated with $\lambda$.

---

## What is a linear PDE?

A PDE is **linear** if the unknown $u$ and all its derivatives appear only to the **first power**
with no products between them. Equivalently, the operator $\mathcal{L}$ acting on $u$ satisfies:

$$\mathcal{L}[c_1\,u_1 + c_2\,u_2] = c_1\,\mathcal{L}[u_1] + c_2\,\mathcal{L}[u_2]$$

This is called the **superposition principle**.

**Linear PDEs** — S-L and separation of variables apply:

| PDE | Operator $\mathcal{L}[u]$ | Why linear |
|-----|--------------------------|------------|
| $u_t = \alpha\nabla^2 u$ (heat) | $u_t - \alpha u_{xx}$ | $u$ and its derivatives appear once, no products |
| $\nabla^2 u = 0$ (Laplace) | $u_{xx} + u_{yy}$ | same |
| $u_{tt} = c^2 u_{xx}$ (wave) | $u_{tt} - c^2 u_{xx}$ | same |

**Non-linear PDEs** — superposition fails, S-L does not apply:

| PDE | Non-linear term | Why non-linear |
|-----|----------------|----------------|
| $u_t + u\,u_x = \nu u_{xx}$ (Burgers) | $u\,u_x$ | $u$ multiplied by its own derivative |
| $u_t + (u\cdot\nabla)u = \ldots$ (Navier-Stokes) | $(u\cdot\nabla)u$ | same — convective acceleration |
| $u_t + 6u\,u_x + u_{xxx} = 0$ (KdV) | $u\,u_x$ | same |

**Why linearity is essential for separation of variables:**  
After separation you find individual product solutions $X_n(x)\,Y_n(y)$. The general solution requires summing them:

$$u = \sum_n a_n\,X_n(x)\,Y_n(y)$$

This sum is itself a solution **only because the PDE is linear** — linearity is exactly the superposition
principle that lets you add solutions freely. In a non-linear PDE, the sum of two solutions is
generally not a solution.

---

## Connection to the 2D Laplace plate problem

The $X$-equation from the heat plate notebook was:

$$X'' + \lambda X = 0, \quad X'(0)=0,\; X'(L)=0$$

This is the S-L problem with $p=1$, $q=0$, $w=1$ — the simplest possible case.

---

## The three guarantees S-L theory provides

These are theorems — they make separation of variables rigorous rather than just a lucky trick.

1. **Real, discrete eigenvalues** — $\lambda_0 < \lambda_1 < \lambda_2 < \cdots \to +\infty$.  
   Only countably many separation constants actually work.

2. **Orthogonality** — eigenfunctions $\phi_m$, $\phi_n$ for different eigenvalues satisfy:
$$\int_a^b \phi_m(x)\,\phi_n(x)\,w(x)\,dx = 0 \quad (m \neq n)$$
   This is what lets you isolate Fourier coefficients by integrating.

3. **Completeness** — any piecewise-smooth $f(x)$ has a convergent expansion:
$$f(x) = \sum_{n=0}^\infty a_n\,\phi_n(x), \qquad a_n = \frac{\int_a^b f(x)\,\phi_n(x)\,w(x)\,dx}{\int_a^b \phi_n(x)^2\,w(x)\,dx}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.integrate import quad

# ── Finite-difference discretisation of the S-L operator ─────────────────────
# Discretise [a, b] into N interior points (Dirichlet ends) or N+1 (Neumann)
# We use a general finite-difference matrix approach.

def build_sl_matrix(p_fn, q_fn, w_fn, a, b, N, bc="dirichlet"):
    """
    Build the finite-difference matrix for the S-L eigenvalue problem
        -(p y')' + q y = λ w y
    on [a, b] with N interior grid points.

    Returns (A, W, x) where A φ = λ W φ  (generalised eigenproblem).

    bc = "dirichlet" : y(a)=y(b)=0
    bc = "neumann"   : y'(a)=y'(b)=0  (ghost-point method)
    """
    if bc == "dirichlet":
        x = np.linspace(a, b, N + 2)[1:-1]   # interior points only
    else:
        x = np.linspace(a, b, N + 2)[1:-1]   # same grid; BCs handled below

    h = x[1] - x[0]

    # Evaluate coefficient functions on the grid
    p = p_fn(x)
    q = q_fn(x)
    w = w_fn(x)

    # p at half-points using arithmetic mean
    p_half_right = 0.5 * (p[:-1] + p[1:])   # p_{i+1/2}
    p_half_left  = 0.5 * (p[1:]  + p[:-1])  # p_{i-1/2}  (same values, shifted index)

    # Stiffness matrix A for -(p y')'  + q y  (central differences)
    main_diag = (np.roll(p, -1) + 2*p + np.roll(p, 1)) / (2 * h**2) + q
    # More careful: use half-point values
    diag_main  = np.zeros(N)
    diag_upper = np.zeros(N - 1)
    diag_lower = np.zeros(N - 1)

    for i in range(N):
        p_left  = 0.5 * (p[i-1] + p[i])  if i > 0   else p[0]
        p_right = 0.5 * (p[i]   + p[i+1]) if i < N-1 else p[-1]

        diag_main[i] = (p_left + p_right) / h**2 + q[i]
        if i < N - 1:
            diag_upper[i] = -p_right / h**2
        if i > 0:
            diag_lower[i-1] = -p_left / h**2

    A = (np.diag(diag_main)
       + np.diag(diag_upper, k=1)
       + np.diag(diag_lower, k=-1))

    # Neumann BC: zero-flux at boundaries via ghost points
    # y'(a)=0  →  y[-1] = y[1]   →  modify first row
    # y'(b)=0  →  y[N]  = y[N-2] →  modify last  row
    if bc == "neumann":
        p_edge_left  = p[0]
        p_edge_right = p[-1]
        A[0,  0]  -= p_edge_left  / h**2   # remove ghost contribution
        A[-1, -1] -= p_edge_right / h**2

    # Weight matrix W = diag(w)
    W = np.diag(w)

    return A, W, x


# ── Helper: solve S-L and return sorted eigenpairs ────────────────────────────
def solve_sl(p_fn, q_fn, w_fn, a, b, N=300, bc="dirichlet", n_modes=8):
    A, W, x = build_sl_matrix(p_fn, q_fn, w_fn, a, b, N, bc)
    vals, vecs = eigh(A, W)          # solves A φ = λ W φ
    idx = np.argsort(vals)
    vals = vals[idx[:n_modes]]
    vecs = vecs[:, idx[:n_modes]]
    # Normalise each eigenfunction: ∫ φ² w dx ≈ 1
    h = x[1] - x[0]
    w_vals = w_fn(x)
    for k in range(n_modes):
        norm = np.sqrt(np.sum(vecs[:, k]**2 * w_vals) * h)
        vecs[:, k] /= norm
        if vecs[1, k] < 0:           # consistent sign convention
            vecs[:, k] *= -1
    return vals, vecs, x

print("Helpers ready.")
print("Grid size N=300, solving generalised eigenproblem  A φ = λ W φ  via scipy.linalg.eigh")

## Case 1 — The simplest S-L problem: $y'' + \lambda y = 0$
### ($p=1,\; q=0,\; w=1$) — exactly the plate problem X-equation

**Dirichlet BCs** $y(0)=y(1)=0$: eigenfunctions are $\sin(n\pi x)$, eigenvalues $\lambda_n = (n\pi)^2$  
**Neumann BCs** $y'(0)=y'(1)=0$: eigenfunctions are $\cos(n\pi x)$, eigenvalues $\lambda_n = (n\pi)^2$

We compare numerical results against the exact analytical values.

In [ ]:
p1 = lambda x: np.ones_like(x)
q1 = lambda x: np.zeros_like(x)
w1 = lambda x: np.ones_like(x)
a, b = 0.0, 1.0
n_modes = 6

vals_D, vecs_D, x_D = solve_sl(p1, q1, w1, a, b, N=300, bc="dirichlet", n_modes=n_modes)
vals_N, vecs_N, x_N = solve_sl(p1, q1, w1, a, b, N=300, bc="neumann",   n_modes=n_modes)

# ── Exact eigenvalues ─────────────────────────────────────────────────────────
exact_D = [(n * np.pi)**2 for n in range(1, n_modes + 1)]
exact_N = [(n * np.pi)**2 for n in range(0, n_modes)]

print("Dirichlet  (sin modes):  y(0)=y(1)=0")
print(f"  {'n':>3}  {'λ_n exact':>14}  {'λ_n numeric':>14}  {'error':>10}")
for n, (ex, nu) in enumerate(zip(exact_D, vals_D), start=1):
    print(f"  {n:>3}  {ex:>14.6f}  {nu:>14.6f}  {abs(ex-nu):>10.2e}")

print("\nNeumann  (cos modes):  y'(0)=y'(1)=0")
print(f"  {'n':>3}  {'λ_n exact':>14}  {'λ_n numeric':>14}  {'error':>10}")
for n, (ex, nu) in enumerate(zip(exact_N, vals_N)):
    print(f"  {n:>3}  {ex:>14.6f}  {nu:>14.6f}  {abs(ex-nu):>10.2e}")

# ── Plot eigenfunctions ───────────────────────────────────────────────────────
fig, axes = plt.subplots(2, n_modes, figsize=(16, 5), sharey=False)
x_fine = np.linspace(a, b, 500)

for k in range(n_modes):
    # Dirichlet
    exact_fn_D = np.sqrt(2) * np.sin((k+1) * np.pi * x_fine)
    axes[0, k].plot(x_fine, exact_fn_D, 'k--', lw=1.5, label='exact')
    axes[0, k].plot(x_D, vecs_D[:, k], color='steelblue', lw=2, label='numeric')
    axes[0, k].set_title(f"$n={k+1}$\n$\\lambda={vals_D[k]:.2f}$", fontsize=9)
    axes[0, k].axhline(0, color='gray', lw=0.5)
    axes[0, k].set_xticks([0, 0.5, 1])
    if k == 0:
        axes[0, k].set_ylabel("Dirichlet\n$\\phi_n(x)$")

    # Neumann
    exact_fn_N = np.ones_like(x_fine) if k == 0 else np.sqrt(2) * np.cos(k * np.pi * x_fine)
    axes[1, k].plot(x_fine, exact_fn_N, 'k--', lw=1.5)
    axes[1, k].plot(x_N, vecs_N[:, k], color='tomato', lw=2)
    axes[1, k].set_title(f"$n={k}$\n$\\lambda={vals_N[k]:.2f}$", fontsize=9)
    axes[1, k].axhline(0, color='gray', lw=0.5)
    axes[1, k].set_xticks([0, 0.5, 1])
    if k == 0:
        axes[1, k].set_ylabel("Neumann\n$\\phi_n(x)$")

axes[0, 0].legend(fontsize=7)
fig.suptitle(r"Case 1: $y'' + \lambda y = 0$  —  Dirichlet (sin) vs Neumann (cos) eigenfunctions",
             fontsize=12)
plt.tight_layout()
plt.show()

## Case 2 — Variable $p(x)$: non-uniform stiffness

Now set $p(x) = 1 + 3x^2$ (stiffness increases towards the right), $q=0$, $w=1$, Dirichlet BCs.

The eigenfunctions are **no longer pure sines** — they get compressed where $p$ is large (stiffer
medium) and stretched where $p$ is small. The eigenvalues are no longer $(n\pi)^2$ but are still
real, discrete, and increasing. S-L theory guarantees this even when there is no closed-form solution.

In [ ]:
p2 = lambda x: 1 + 3 * x**2      # variable stiffness
q2 = lambda x: np.zeros_like(x)
w2 = lambda x: np.ones_like(x)

# Compare: uniform p=1 vs variable p=1+3x²
vals_unif, vecs_unif, x_u = solve_sl(p1, q2, w2, 0, 1, N=300, bc="dirichlet", n_modes=5)
vals_var,  vecs_var,  x_v = solve_sl(p2, q2, w2, 0, 1, N=300, bc="dirichlet", n_modes=5)

print("Eigenvalue comparison:")
print(f"  {'n':>3}  {'λ_n  (p=1)':>14}  {'λ_n  (p=1+3x²)':>18}  {'ratio':>8}")
for n, (lu, lv) in enumerate(zip(vals_unif, vals_var), start=1):
    print(f"  {n:>3}  {lu:>14.4f}  {lv:>18.4f}  {lv/lu:>8.4f}")

fig, axes = plt.subplots(1, 5, figsize=(16, 4), sharey=True)
x_fine = np.linspace(0, 1, 500)

for k in range(5):
    axes[k].plot(x_u, vecs_unif[:, k], 'k--', lw=1.5, label=r'$p=1$ (sine)')
    axes[k].plot(x_v, vecs_var[:, k],  color='darkorange', lw=2,
                 label=r'$p=1+3x^2$')
    axes[k].axhline(0, color='gray', lw=0.5)
    axes[k].set_title(f"$n={k+1}$\n"
                      f"$\\lambda={vals_unif[k]:.1f}\\to{vals_var[k]:.1f}$", fontsize=9)
    axes[k].set_xlabel("$x$")
    if k == 0:
        axes[k].set_ylabel("$\\phi_n(x)$")
        axes[k].legend(fontsize=7)

# Overlay p(x) on the last panel to show where stiffness is high
ax_twin = axes[-1].twinx()
ax_twin.fill_between(x_fine, 0, p2(x_fine), alpha=0.15, color='darkorange')
ax_twin.plot(x_fine, p2(x_fine), color='darkorange', lw=1, ls=':')
ax_twin.set_ylabel("$p(x) = 1+3x^2$", color='darkorange', fontsize=8)
ax_twin.tick_params(axis='y', labelcolor='darkorange')

fig.suptitle(r"Case 2: variable $p(x)=1+3x^2$ — eigenfunctions shift towards the softer (left) side",
             fontsize=12)
plt.tight_layout()
plt.show()

## Verifying Orthogonality

S-L theory guarantees $\displaystyle\int_a^b \phi_m(x)\,\phi_n(x)\,w(x)\,dx = 0$ for $m \neq n$.

We check this numerically for both the uniform ($p=1$) and variable ($p=1+3x^2$) cases.
The result should be an **identity matrix** (since we also normalised each eigenfunction).

In [ ]:
def gram_matrix(vecs, x, w_fn):
    """Compute the weighted inner-product matrix  G[m,n] = ∫ φ_m φ_n w dx."""
    h = x[1] - x[0]
    w = w_fn(x)
    n_modes = vecs.shape[1]
    G = np.zeros((n_modes, n_modes))
    for m in range(n_modes):
        for n in range(n_modes):
            G[m, n] = np.sum(vecs[:, m] * vecs[:, n] * w) * h
    return G

n_check = 6

# Re-solve with more modes for the check
vals_u6, vecs_u6, x_u6 = solve_sl(p1, q2, w2, 0, 1, N=300, bc="dirichlet", n_modes=n_check)
vals_v6, vecs_v6, x_v6 = solve_sl(p2, q2, w2, 0, 1, N=300, bc="dirichlet", n_modes=n_check)

G_unif = gram_matrix(vecs_u6, x_u6, w2)
G_var  = gram_matrix(vecs_v6, x_v6, w2)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, G, title in zip(axes,
                         [G_unif, G_var],
                         [r"$p=1$,  $w=1$  (uniform)",
                          r"$p=1+3x^2$,  $w=1$  (variable)"]):
    im = ax.imshow(np.abs(G), cmap='Blues', vmin=0, vmax=1.05)
    plt.colorbar(im, ax=ax)
    # Annotate each cell
    for i in range(n_check):
        for j in range(n_check):
            ax.text(j, i, f"{G[i,j]:.2f}", ha='center', va='center',
                    fontsize=8, color='black' if abs(G[i,j]) < 0.6 else 'white')
    ax.set_title(f"Gram matrix  $G_{{mn}} = \\langle\\phi_m,\\phi_n\\rangle_w$\n{title}")
    ax.set_xlabel("$n$"); ax.set_ylabel("$m$")
    ax.set_xticks(range(n_check)); ax.set_yticks(range(n_check))

plt.suptitle("Orthogonality check — should be identity matrix", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nMax off-diagonal element (uniform p): {np.max(np.abs(G_unif - np.eye(n_check))):.2e}")
print(f"Max off-diagonal element (variable p): {np.max(np.abs(G_var  - np.eye(n_check))):.2e}")

## Eigenfunction Expansion (Generalised Fourier Series)

Given any target function $f(x)$, S-L theory guarantees we can write:

$$f(x) = \sum_{n=0}^{\infty} a_n\,\phi_n(x), \qquad
  a_n = \frac{\displaystyle\int_a^b f(x)\,\phi_n(x)\,w(x)\,dx}
             {\displaystyle\int_a^b \phi_n(x)^2\,w(x)\,dx}$$

This is exactly what happens in the plate problem: the top BC $f(x)$ is expanded in the
cosine eigenfunctions to find the coefficients $a_n$.

Below we expand a **step function** (challenging — not smooth) in the eigenfunctions of both
the uniform and variable-$p$ S-L problems, and watch the series converge.

In [ ]:
def sl_expand(f_vals, vecs, x, w_fn):
    """
    Compute S-L expansion coefficients  a_n = <f, φ_n>_w / <φ_n, φ_n>_w.
    Because eigenfunctions are already normalised,  <φ_n, φ_n>_w = 1.
    """
    h   = x[1] - x[0]
    w   = w_fn(x)
    return np.array([np.sum(f_vals * vecs[:, n] * w) * h
                     for n in range(vecs.shape[1])])


# ── Target function: step ────────────────────────────────────────────────────
n_modes_exp = 30
vals_u30, vecs_u30, x30 = solve_sl(p1, q2, w2, 0, 1, N=500, bc="dirichlet", n_modes=n_modes_exp)
vals_v30, vecs_v30, _   = solve_sl(p2, q2, w2, 0, 1, N=500, bc="dirichlet", n_modes=n_modes_exp)

f_step = np.where(x30 < 0.5, 1.0, 0.0).astype(float)

coeffs_u = sl_expand(f_step, vecs_u30, x30, w2)
coeffs_v = sl_expand(f_step, vecs_v30, x30, w2)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
truncations = [1, 3, 6, 30]
colors = ['steelblue', 'darkorange']

for row, (vecs30, coeffs, label, color) in enumerate([
        (vecs_u30, coeffs_u, r"$p=1$ (sine basis)",        'steelblue'),
        (vecs_v30, coeffs_v, r"$p=1+3x^2$ (S-L basis)", 'darkorange')]):

    for col, N_trunc in enumerate(truncations):
        f_approx = vecs30[:, :N_trunc] @ coeffs[:N_trunc]
        ax = axes[row, col]
        ax.plot(x30, f_step,   'k-',  lw=1.5, label='target $f$')
        ax.plot(x30, f_approx, color=color, lw=2,
                label=f'{N_trunc} modes')
        ax.set_ylim(-0.3, 1.4)
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_title(f"{N_trunc} mode{'s' if N_trunc>1 else ''}", fontsize=10)
        ax.set_xlabel("$x$")
        if col == 0:
            ax.set_ylabel(label, fontsize=9)
        if col == 0 and row == 0:
            ax.legend(fontsize=7)

fig.suptitle("Eigenfunction expansion of a step function — convergence with more modes",
             fontsize=12)
plt.tight_layout()
plt.show()

# ── Coefficient spectrum ──────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
for ax, coeffs, label, color in zip(axes2,
        [coeffs_u, coeffs_v],
        [r"$p=1$ basis", r"$p=1+3x^2$ basis"],
        ['steelblue', 'darkorange']):
    ax.stem(range(n_modes_exp), np.abs(coeffs),
            linefmt=color, markerfmt='o', basefmt='k-')
    ax.set_title(f"Expansion coefficients  $|a_n|$ — {label}")
    ax.set_xlabel("mode $n$"); ax.set_ylabel("$|a_n|$")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Case 3 — Other classic S-L problems

Different PDEs and coordinate systems give different $p$, $q$, $w$ — but the same S-L structure.

### Bessel's equation (appears when solving PDEs on a disk in polar coordinates)

$$-(x\,y')' + \frac{n^2}{x}\,y = \lambda\,x\,y \quad \Rightarrow \quad p(x)=x,\; q(x)=n^2/x,\; w(x)=x$$

Eigenfunctions are **Bessel functions** $J_n(\sqrt{\lambda}\,x)$ — the "circular" generalization of sines.

### Legendre's equation (appears on the sphere)

$$-\left[(1-x^2)\,y'\right]' = \lambda\,y \quad \Rightarrow \quad p(x)=1-x^2,\; q=0,\; w=1 \quad \text{on } [-1,1]$$

Eigenfunctions are **Legendre polynomials** $P_n(x)$, eigenvalues $\lambda_n = n(n+1)$.

We compute both numerically and compare to the known closed-form solutions.

In [ ]:
from scipy.special import jn, eval_legendre

# ── Bessel  (n=0 order, first kind)  on [ε, 1] with Dirichlet BCs ────────────
eps = 1e-3   # avoid singularity at x=0
p_bes = lambda x: x
q_bes = lambda x: np.zeros_like(x)   # Bessel order 0 → q = n²/x = 0
w_bes = lambda x: x                  # weight = x  (polar area element)

vals_bes, vecs_bes, x_bes = solve_sl(p_bes, q_bes, w_bes,
                                      eps, 1.0, N=400, bc="dirichlet", n_modes=5)

# Exact: zeros of J₀ give eigenvalues λ_k = z_k²  where J₀(z_k)=0
j0_zeros = np.array([2.4048, 5.5201, 8.6537, 11.7915, 14.9309])
exact_bes = j0_zeros**2

print("Bessel (order 0) eigenvalues:")
print(f"  {'k':>3}  {'λ_k exact (z_k²)':>18}  {'λ_k numeric':>14}  {'error':>10}")
for k, (ex, nu) in enumerate(zip(exact_bes, vals_bes), start=1):
    print(f"  {k:>3}  {ex:>18.4f}  {nu:>14.4f}  {abs(ex-nu):>10.2e}")

# ── Legendre  on [-1+ε, 1-ε] ─────────────────────────────────────────────────
eps2 = 1e-3
p_leg = lambda x: 1 - x**2
q_leg = lambda x: np.zeros_like(x)
w_leg = lambda x: np.ones_like(x)

vals_leg, vecs_leg, x_leg = solve_sl(p_leg, q_leg, w_leg,
                                      -1+eps2, 1-eps2, N=400, bc="dirichlet", n_modes=6)

exact_leg = [n*(n+1) for n in range(1, 7)]   # n=1,2,3,... (n=0 is constant, excluded by Dirichlet)

print("\nLegendre eigenvalues:")
print(f"  {'n':>3}  {'λ_n = n(n+1)':>14}  {'λ_n numeric':>14}  {'error':>10}")
for n, (ex, nu) in enumerate(zip(exact_leg, vals_leg), start=1):
    print(f"  {n:>3}  {ex:>14.4f}  {nu:>14.4f}  {abs(ex-nu):>10.2e}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
x_fine_bes = np.linspace(eps, 1, 500)
x_fine_leg = np.linspace(-1+eps2, 1-eps2, 500)

for k in range(5):
    # Bessel
    z = j0_zeros[k]
    j0_exact = jn(0, z * x_fine_bes)
    # normalise exact for comparison
    norm_e = np.sqrt(np.trapz(j0_exact**2 * x_fine_bes, x_fine_bes))
    j0_exact /= norm_e
    if j0_exact[10] < 0: j0_exact *= -1

    axes[0, k].plot(x_fine_bes, j0_exact,    'k--', lw=1.5, label='exact $J_0$')
    axes[0, k].plot(x_bes,      vecs_bes[:, k], color='teal', lw=2, label='numeric')
    axes[0, k].set_title(f"$k={k+1}$\n$\\lambda={vals_bes[k]:.2f}$", fontsize=9)
    axes[0, k].axhline(0, color='gray', lw=0.5)
    axes[0, k].set_xlabel("$r$")
    if k == 0:
        axes[0, k].set_ylabel("Bessel $J_0$\n$\\phi_k(r)$")
        axes[0, k].legend(fontsize=7)

for k in range(5):
    # Legendre P_{n} with n = k+1
    n = k + 1
    P_exact = eval_legendre(n, x_fine_leg)
    norm_e = np.sqrt(np.trapz(P_exact**2, x_fine_leg))
    P_exact /= norm_e
    if P_exact[len(P_exact)//2 + 5] * vecs_leg[len(x_leg)//2 + 5, k] < 0:
        P_exact *= -1

    axes[1, k].plot(x_fine_leg, P_exact,       'k--', lw=1.5, label=f'$P_{n}$')
    axes[1, k].plot(x_leg,      vecs_leg[:, k], color='purple', lw=2, label='numeric')
    axes[1, k].set_title(f"$n={n}$\n$\\lambda={n*(n+1)}$", fontsize=9)
    axes[1, k].axhline(0, color='gray', lw=0.5)
    axes[1, k].set_xlabel("$x$")
    if k == 0:
        axes[1, k].set_ylabel("Legendre $P_n$\n$\\phi_n(x)$")
        axes[1, k].legend(fontsize=7)

fig.suptitle("Case 3: Bessel (top) and Legendre (bottom) — all are S-L eigenfunctions",
             fontsize=12)
plt.tight_layout()
plt.show()

## Summary — The S-L Family Tree

Every time you separate variables in a linear PDE, the resulting ODE is a S-L problem.
The coordinate system and domain shape determine which member of the family you get.

| PDE domain | Coordinate system | S-L equation | Eigenfunctions | Used in |
|---|---|---|---|---|
| Rectangle | Cartesian | $y'' + \lambda y = 0$ | $\sin$, $\cos$ | Heat, Laplace on rectangle |
| Disk | Polar ($r$) | $-(ry')' = \lambda r\,y$ | Bessel $J_n$ | Drum vibrations, circular heat |
| Sphere | Spherical ($\theta$) | $-[(1-x^2)y']' = \lambda y$ | Legendre $P_n$ | Gravity, EM on sphere |
| Quantum harmonic oscillator | — | $-y'' + x^2 y = \lambda y$ | Hermite $H_n$ | Quantum mechanics |
| Variable-medium wave | — | $-(p(x)y')' = \lambda w(x) y$ | numerical | Acoustics, elasticity |

### The one unifying idea

Regardless of which row you are in, the same three facts hold:
1. **Eigenvalues are real and discrete** — the separation constant can only take countably many values.
2. **Eigenfunctions are orthogonal** w.r.t. the weight $w(x)$ — this is what makes coefficient extraction (Fourier-style integrals) work.
3. **Eigenfunctions are complete** — any reasonable function on the domain can be expanded in them.

These are not coincidences — they follow from a single algebraic property: the S-L operator
$\mathcal{L}y = -(py')' + qy$ is **self-adjoint** (symmetric) with respect to the $w$-weighted inner product,
exactly like a symmetric matrix has real eigenvalues and orthogonal eigenvectors.